# District-Level ETL — D02 / D03 / D04 / D06 / D10 (Census 2011)

Reads per-state district `.XLSX` files from five Census migration tables via direct XML parsing
and writes one cleaned CSV per table to `data-cleaning/` and `react-app/public/`.

| Folder | Table | Topic | Output |
|---|---|---|---|
| `D-02-ALL/` | D-02 | Duration of residence | `district_duration_residence_flows.csv` |
| `D-03-ALL/` | D-03 | Reason for migration | `district_reason_migration_flows.csv` |
| `D-04-ALL/` | D-04 | Education level | `district_education_levels.csv` |
| `D-06-ALL/` | D-06 | Economic activity | `district_economic_activity.csv` |
| `D-10_ALL/` | D-10 | Marital status | `district_marital_status.csv` |


## 1. Imports, paths & output config


In [ ]:
from __future__ import annotations

import csv
import re
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

# ── Namespace constants used in .xlsx XML ────────────────────────────────────
MAIN_NS    = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
REL_ID_KEY = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT    = Path().resolve()          # data-cleaning/
D02_DIR = ROOT / 'D-02-ALL'
D03_DIR = ROOT / 'D-03-ALL'
D04_DIR = ROOT / 'D-04-ALL'
D06_DIR = ROOT / 'D-06-ALL'
D10_DIR = ROOT / 'D-10_ALL'

OUT_D02_CLEAN  = ROOT / 'district_duration_residence_flows.csv'
OUT_D02_PUBLIC = ROOT.parent / 'react-app' / 'public' / 'district_duration_residence_flows.csv'
OUT_D03_CLEAN  = ROOT / 'district_reason_migration_flows.csv'
OUT_D03_PUBLIC = ROOT.parent / 'react-app' / 'public' / 'district_reason_migration_flows.csv'
OUT_D04_CLEAN  = ROOT / 'district_education_levels.csv'
OUT_D04_PUBLIC = ROOT.parent / 'react-app' / 'public' / 'district_education_levels.csv'
OUT_D06_CLEAN  = ROOT / 'district_economic_activity.csv'
OUT_D06_PUBLIC = ROOT.parent / 'react-app' / 'public' / 'district_economic_activity.csv'
OUT_D10_CLEAN  = ROOT / 'district_marital_status.csv'
OUT_D10_PUBLIC = ROOT.parent / 'react-app' / 'public' / 'district_marital_status.csv'

print('Paths configured.')
print(f'D02 source : {D02_DIR}')
print(f'D03 source : {D03_DIR}')
print(f'D04 source : {D04_DIR}')
print(f'D06 source : {D06_DIR}')
print(f'D10 source : {D10_DIR}')


## 2. State & district reference sets


In [ ]:
# Canonical state/UT names — used to validate both origin and destination
STATE_NAMES = {
    "JAMMU & KASHMIR", "HIMACHAL PRADESH", "PUNJAB", "CHANDIGARH", "UTTARAKHAND",
    "HARYANA", "NCT OF DELHI", "RAJASTHAN", "UTTAR PRADESH", "BIHAR", "SIKKIM",
    "ARUNACHAL PRADESH", "NAGALAND", "MANIPUR", "MIZORAM", "TRIPURA", "MEGHALAYA",
    "ASSAM", "WEST BENGAL", "JHARKHAND", "ODISHA", "CHHATTISGARH", "MADHYA PRADESH",
    "GUJARAT", "DAMAN & DIU", "DADRA & NAGAR HAVELI", "MAHARASHTRA", "ANDHRA PRADESH",
    "KARNATAKA", "GOA", "LAKSHADWEEP", "KERALA", "TAMIL NADU", "PUDUCHERRY",
    "ANDAMAN & NICOBAR ISLANDS", "TELANGANA",
}

# Districts that appear under Andhra Pradesh in 2011 data but belong to Telangana
TELANGANA_DISTRICTS = {
    "ADILABAD", "NIZAMABAD", "KARIMNAGAR", "MEDAK", "WARANGAL",
    "RANGAREDDI", "RANGAREDDY", "HYDERABAD", "NALGONDA", "KHAMMAM", "MAHBUBNAGAR",
}

print(f'State/UT count : {len(STATE_NAMES)}')
print(f'Telangana districts tracked : {len(TELANGANA_DISTRICTS)}')


## 3. Output column headers


In [ ]:
D02_HEADERS = [
    "state", "district", "districtCode", "origin",
    "Persons_Total",    "Males_Total",    "Females_Total",
    "Persons_LT1yr",    "Males_LT1yr",    "Females_LT1yr",
    "Persons_1to4yr",   "Males_1to4yr",   "Females_1to4yr",
    "Persons_5to9yr",   "Males_5to9yr",   "Females_5to9yr",
    "Persons_10to19yr", "Males_10to19yr", "Females_10to19yr",
    "Persons_20plusyr", "Males_20plusyr", "Females_20plusyr",
    "Persons_DurNS",    "Males_DurNS",    "Females_DurNS",
]

D03_HEADERS = [
    "state", "district", "districtCode", "origin",
    "Persons_Total",          "Males_Total",          "Females_Total",
    "Persons_Work",           "Males_Work",           "Females_Work",
    "Persons_Business",       "Males_Business",       "Females_Business",
    "Persons_Education",      "Males_Education",      "Females_Education",
    "Persons_Marriage",       "Males_Marriage",       "Females_Marriage",
    "Persons_MoveAfterBirth", "Males_MoveAfterBirth", "Females_MoveAfterBirth",
    "Persons_MoveWithHH",     "Males_MoveWithHH",     "Females_MoveWithHH",
    "Persons_Other",          "Males_Other",          "Females_Other",
]

D04_HEADERS = [
    "state", "district", "districtCode",
    "Illiterate_Persons",   "Illiterate_Males",   "Illiterate_Females",
    "Literate_Persons",     "Literate_Males",     "Literate_Females",
    "BelowMatric_Persons",  "BelowMatric_Males",  "BelowMatric_Females",
    "MatricToGrad_Persons", "MatricToGrad_Males", "MatricToGrad_Females",
    "TechDiploma_Persons",  "TechDiploma_Males",  "TechDiploma_Females",
    "Graduate_Persons",     "Graduate_Males",     "Graduate_Females",
    "TechDegree_Persons",   "TechDegree_Males",   "TechDegree_Females",
]

D06_HEADERS = [
    "state", "district", "districtCode",
    "MainWorkers_Persons",      "MainWorkers_Males",      "MainWorkers_Females",
    "MarginalWorkers_Persons",  "MarginalWorkers_Males",  "MarginalWorkers_Females",
    "MarginalSeeking_Persons",  "MarginalSeeking_Males",  "MarginalSeeking_Females",
    "NonWorkers_Persons",       "NonWorkers_Males",       "NonWorkers_Females",
    "NonWorkersSeeking_Persons","NonWorkersSeeking_Males","NonWorkersSeeking_Females",
]

D10_HEADERS = [
    "state", "district", "districtCode",
    "NeverMarried_Persons", "NeverMarried_Males", "NeverMarried_Females",
    "CurrMarried_Persons",  "CurrMarried_Males",  "CurrMarried_Females",
    "Widowed_Persons",      "Widowed_Males",      "Widowed_Females",
    "Separated_Persons",    "Separated_Males",    "Separated_Females",
    "Divorced_Persons",     "Divorced_Males",     "Divorced_Females",
    "Unspecified_Persons",  "Unspecified_Males",  "Unspecified_Females",
]

print('Column headers defined.')


## 4. Low-level XLSX parser


In [ ]:
def to_int(value) -> int:
    """Safely convert any cell value to int (returns 0 on failure)."""
    if value is None:
        return 0
    text = str(value).strip()
    if not text:
        return 0
    try:
        return int(float(text))
    except (TypeError, ValueError):
        return 0


def normalize_name(name) -> str:
    """Uppercase, strip prefixes, and apply canonical name mapping."""
    if name is None:
        return ""
    value = str(name).upper().strip()
    value = value.replace("UNION TERRITORY - ", "")
    value = value.replace("STATE - ", "")
    value = value.replace("UT - ", "")
    mapping = {
        "DELHI":                         "NCT OF DELHI",
        "ORISSA":                        "ODISHA",
        "TELENGANA":                     "TELANGANA",
        "UTTARANCHAL":                   "UTTARAKHAND",
        "JAMMU AND KASHMIR":             "JAMMU & KASHMIR",
        "ANDAMAN AND NICOBAR":           "ANDAMAN & NICOBAR ISLANDS",
        "ANDAMAN AND NICOBAR ISLANDS":   "ANDAMAN & NICOBAR ISLANDS",
        "ANDAMAN & NICOBAR":             "ANDAMAN & NICOBAR ISLANDS",
        "DADRA AND NAGAR HAVELI":        "DADRA & NAGAR HAVELI",
        "DAMAN AND DIU":                 "DAMAN & DIU",
    }
    return mapping.get(value, value)


def normalize_flag(value) -> str:
    """Strip and uppercase a filter column value for comparison."""
    return str(value or "").strip().upper()


def looks_like_state_area_name(value: str) -> bool:
    """True if AreaName is a state/UT header row, not a district row."""
    text = value.upper().strip()
    return text.startswith("STATE -") or text.startswith("UNION TERRITORY -") or text.startswith("UT -")


def parse_relationships(xml_bytes: bytes) -> dict:
    root = ET.fromstring(xml_bytes)
    return {rel.attrib.get('Id', ''): rel.attrib.get('Target', '') for rel in root}


def read_shared_strings(zf: zipfile.ZipFile) -> list:
    if "xl/sharedStrings.xml" not in zf.namelist():
        return []
    root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
    values = []
    for si in root.findall(f'{MAIN_NS}si'):
        parts = [t.text or '' for t in si.findall(f'.//{MAIN_NS}t')]
        values.append("".join(parts))
    return values


def get_sheet_path(zf: zipfile.ZipFile, preferred_sheet_name: str) -> str:
    """Return the internal XML path for the named worksheet."""
    wb = ET.fromstring(zf.read("xl/workbook.xml"))
    relation_map = parse_relationships(zf.read("xl/_rels/workbook.xml.rels"))
    chosen = fallback = None
    for sheet in wb.findall(f'.//{MAIN_NS}sheet'):
        rid    = sheet.attrib.get(REL_ID_KEY, '')
        target = relation_map.get(rid, '')
        if not target:
            continue
        path = target[1:] if target.startswith('/') else f'xl/{target}'
        if fallback is None:
            fallback = path
        if str(sheet.attrib.get('name', '')).strip().upper() == preferred_sheet_name.upper():
            chosen = path
            break
    if chosen:
        return chosen
    if fallback:
        return fallback
    raise ValueError("No worksheet found in workbook")


def column_index(cell_ref: str) -> int:
    """Convert Excel column letters (A, B, AA …) to 0-based index."""
    letters = "".join(ch for ch in cell_ref if ch.isalpha())
    index = 0
    for ch in letters:
        index = (index * 26) + (ord(ch) - 64)
    return index - 1


def read_cell_value(cell: ET.Element, shared_strings: list) -> str:
    """Return the string value of a cell, resolving shared string references."""
    cell_type = cell.attrib.get('t', '')
    if cell_type == 'inlineStr':
        is_node = cell.find(f'{MAIN_NS}is')
        if is_node is None:
            return ""
        parts = [t.text or '' for t in is_node.findall(f'.//{MAIN_NS}t')]
        return "".join(parts).strip()
    value_node = cell.find(f'{MAIN_NS}v')
    if value_node is None:
        return ""
    raw = (value_node.text or '').strip()
    if cell_type == 's':
        try:
            return shared_strings[int(raw)].strip()
        except (ValueError, IndexError):
            return raw
    return raw


def iter_sheet_rows(workbook_path: Path, sheet_name: str):
    """Yield each data row as a dict of {column_index: cell_value}."""
    with zipfile.ZipFile(workbook_path) as zf:
        shared_strings = read_shared_strings(zf)
        sheet_path = get_sheet_path(zf, sheet_name)
        worksheet = ET.fromstring(zf.read(sheet_path))
        for row in worksheet.findall(f'.//{MAIN_NS}sheetData/{MAIN_NS}row'):
            cells = {}
            for cell in row.findall(f'{MAIN_NS}c'):
                ref = cell.attrib.get('r', '')
                if ref:
                    cells[column_index(ref)] = read_cell_value(cell, shared_strings)
            yield cells


print('Parser functions defined.')


## 5. Extract functions (D02 / D03 / D04 / D06 / D10)


In [ ]:
def extract_d02_rows() -> list:
    """D-02: Last Residence x Duration — one row per (district, origin state) pair."""
    output = []
    for workbook in sorted(D02_DIR.glob('*.XLSX')):
        workbook_state = ''
        for row in iter_sheet_rows(workbook, 'D-02'):
            area_name = str(row.get(3, '')).strip()
            if not workbook_state and looks_like_state_area_name(area_name):
                workbook_state = normalize_name(area_name)
            district_code = to_int(row.get(2))
            if district_code == 0:
                continue
            if not area_name or looks_like_state_area_name(area_name):
                continue
            # Keep only Total x Total TRU rows
            if normalize_flag(row.get(5, '')) != 'TOTAL' or normalize_flag(row.get(6, '')) != 'TOTAL':
                continue
            origin_state = normalize_name(row.get(4, ''))
            if origin_state not in STATE_NAMES:
                continue
            if to_int(row.get(7)) <= 0:
                continue
            display_state = 'TELANGANA' if area_name.upper() in TELANGANA_DISTRICTS else workbook_state
            if display_state not in STATE_NAMES:
                continue
            output.append({
                'state': display_state, 'district': area_name, 'districtCode': district_code,
                'origin': origin_state,
                'Persons_Total':    to_int(row.get(7)),  'Males_Total':    to_int(row.get(8)),  'Females_Total':    to_int(row.get(9)),
                'Persons_LT1yr':    to_int(row.get(10)), 'Males_LT1yr':    to_int(row.get(11)), 'Females_LT1yr':    to_int(row.get(12)),
                'Persons_1to4yr':   to_int(row.get(13)), 'Males_1to4yr':   to_int(row.get(14)), 'Females_1to4yr':   to_int(row.get(15)),
                'Persons_5to9yr':   to_int(row.get(16)), 'Males_5to9yr':   to_int(row.get(17)), 'Females_5to9yr':   to_int(row.get(18)),
                'Persons_10to19yr': to_int(row.get(19)), 'Males_10to19yr': to_int(row.get(20)), 'Females_10to19yr': to_int(row.get(21)),
                'Persons_20plusyr': to_int(row.get(22)), 'Males_20plusyr': to_int(row.get(23)), 'Females_20plusyr': to_int(row.get(24)),
                'Persons_DurNS':    to_int(row.get(25)), 'Males_DurNS':    to_int(row.get(26)), 'Females_DurNS':    to_int(row.get(27)),
            })
    output.sort(key=lambda r: (r['state'], r['district'], -int(r['Persons_Total'])))
    return output


def extract_d03_rows() -> list:
    """D-03: Last Residence x Reason for Migration — one row per (district, origin state) pair."""
    output = []
    for workbook in sorted(D03_DIR.glob('*.XLSX')):
        workbook_state = ''
        for row in iter_sheet_rows(workbook, 'D-03'):
            area_name = str(row.get(3, '')).strip()
            if not workbook_state and looks_like_state_area_name(area_name):
                workbook_state = normalize_name(area_name)
            district_code = to_int(row.get(2))
            if district_code == 0:
                continue
            if not area_name or looks_like_state_area_name(area_name):
                continue
            if normalize_flag(row.get(4, '')) != 'TOTAL':
                continue
            if normalize_flag(row.get(5, '')) != 'ALL DURATIONS OF RESIDENCE':
                continue
            if normalize_flag(row.get(7, '')) != 'TOTAL':
                continue
            origin_state = normalize_name(row.get(6, ''))
            if origin_state not in STATE_NAMES:
                continue
            if to_int(row.get(8)) <= 0:
                continue
            display_state = 'TELANGANA' if area_name.upper() in TELANGANA_DISTRICTS else workbook_state
            if display_state not in STATE_NAMES:
                continue
            output.append({
                'state': display_state, 'district': area_name, 'districtCode': district_code,
                'origin': origin_state,
                'Persons_Total':          to_int(row.get(8)),  'Males_Total':          to_int(row.get(9)),  'Females_Total':          to_int(row.get(10)),
                'Persons_Work':           to_int(row.get(11)), 'Males_Work':           to_int(row.get(12)), 'Females_Work':           to_int(row.get(13)),
                'Persons_Business':       to_int(row.get(14)), 'Males_Business':       to_int(row.get(15)), 'Females_Business':       to_int(row.get(16)),
                'Persons_Education':      to_int(row.get(17)), 'Males_Education':      to_int(row.get(18)), 'Females_Education':      to_int(row.get(19)),
                'Persons_Marriage':       to_int(row.get(20)), 'Males_Marriage':       to_int(row.get(21)), 'Females_Marriage':       to_int(row.get(22)),
                'Persons_MoveAfterBirth': to_int(row.get(23)), 'Males_MoveAfterBirth': to_int(row.get(24)), 'Females_MoveAfterBirth': to_int(row.get(25)),
                'Persons_MoveWithHH':     to_int(row.get(26)), 'Males_MoveWithHH':     to_int(row.get(27)), 'Females_MoveWithHH':     to_int(row.get(28)),
                'Persons_Other':          to_int(row.get(29)), 'Males_Other':          to_int(row.get(30)), 'Females_Other':          to_int(row.get(31)),
            })
    output.sort(key=lambda r: (r['state'], r['district'], -int(r['Persons_Total'])))
    return output


def extract_d04_rows() -> list:
    """D-04: Education level — one row per district (all-durations, all-ages total only)."""
    by_district = {}
    for workbook in sorted(D04_DIR.glob('*.XLSX')):
        workbook_state = ''
        for row in iter_sheet_rows(workbook, 'D-04'):
            area_name = str(row.get(3, '')).strip()
            if not workbook_state and looks_like_state_area_name(area_name):
                workbook_state = normalize_name(area_name)
            district_code = to_int(row.get(2))
            if district_code == 0:
                continue
            if not area_name or looks_like_state_area_name(area_name):
                continue
            if normalize_flag(row.get(4, '')) != 'TOTAL':
                continue
            if normalize_flag(row.get(5, '')) != 'ALL DURATIONS OF RESIDENCE':
                continue
            if normalize_flag(row.get(6, '')) != 'ALL AGES':
                continue
            if normalize_flag(row.get(7, '')) != 'TOTAL':
                continue
            display_state = 'TELANGANA' if area_name.upper() in TELANGANA_DISTRICTS else workbook_state
            if display_state not in STATE_NAMES:
                continue
            key = (display_state, area_name, district_code)
            if key in by_district:
                continue   # first occurrence only
            by_district[key] = {
                'state': display_state, 'district': area_name, 'districtCode': district_code,
                'Illiterate_Persons':   to_int(row.get(11)), 'Illiterate_Males':   to_int(row.get(12)), 'Illiterate_Females':   to_int(row.get(13)),
                'Literate_Persons':     to_int(row.get(14)), 'Literate_Males':     to_int(row.get(15)), 'Literate_Females':     to_int(row.get(16)),
                'BelowMatric_Persons':  to_int(row.get(17)), 'BelowMatric_Males':  to_int(row.get(18)), 'BelowMatric_Females':  to_int(row.get(19)),
                'MatricToGrad_Persons': to_int(row.get(20)), 'MatricToGrad_Males': to_int(row.get(21)), 'MatricToGrad_Females': to_int(row.get(22)),
                'TechDiploma_Persons':  to_int(row.get(23)), 'TechDiploma_Males':  to_int(row.get(24)), 'TechDiploma_Females':  to_int(row.get(25)),
                'Graduate_Persons':     to_int(row.get(26)), 'Graduate_Males':     to_int(row.get(27)), 'Graduate_Females':     to_int(row.get(28)),
                'TechDegree_Persons':   to_int(row.get(29)), 'TechDegree_Males':   to_int(row.get(30)), 'TechDegree_Females':   to_int(row.get(31)),
            }
    output = sorted(by_district.values(), key=lambda r: (r['state'], r['district']))
    return output


def extract_d06_rows() -> list:
    """D-06: Economic activity — one row per district (all-durations, all-ages total only)."""
    by_district = {}
    for workbook in sorted(D06_DIR.glob('*.XLSX')):
        workbook_state = ''
        for row in iter_sheet_rows(workbook, 'D-06'):
            area_name = str(row.get(3, '')).strip()
            if not workbook_state and looks_like_state_area_name(area_name):
                workbook_state = normalize_name(area_name)
            district_code = to_int(row.get(2))
            if district_code == 0:
                continue
            if not area_name or looks_like_state_area_name(area_name):
                continue
            if normalize_flag(row.get(4, '')) != 'TOTAL':
                continue
            if normalize_flag(row.get(5, '')) != 'ALL DURATIONS OF RESIDENCE':
                continue
            if normalize_flag(row.get(6, '')) != 'ALL AGES':
                continue
            if normalize_flag(row.get(7, '')) != 'TOTAL':
                continue
            display_state = 'TELANGANA' if area_name.upper() in TELANGANA_DISTRICTS else workbook_state
            if display_state not in STATE_NAMES:
                continue
            key = (display_state, area_name, district_code)
            if key in by_district:
                continue
            by_district[key] = {
                'state': display_state, 'district': area_name, 'districtCode': district_code,
                'MainWorkers_Persons':       to_int(row.get(11)), 'MainWorkers_Males':       to_int(row.get(12)), 'MainWorkers_Females':       to_int(row.get(13)),
                'MarginalWorkers_Persons':   to_int(row.get(14)), 'MarginalWorkers_Males':   to_int(row.get(15)), 'MarginalWorkers_Females':   to_int(row.get(16)),
                'MarginalSeeking_Persons':   to_int(row.get(17)), 'MarginalSeeking_Males':   to_int(row.get(18)), 'MarginalSeeking_Females':   to_int(row.get(19)),
                'NonWorkers_Persons':        to_int(row.get(20)), 'NonWorkers_Males':        to_int(row.get(21)), 'NonWorkers_Females':        to_int(row.get(22)),
                'NonWorkersSeeking_Persons': to_int(row.get(23)), 'NonWorkersSeeking_Males': to_int(row.get(24)), 'NonWorkersSeeking_Females': to_int(row.get(25)),
            }
    output = sorted(by_district.values(), key=lambda r: (r['state'], r['district']))
    return output


def extract_d10_rows() -> list:
    """D-10: Marital status — one row per district (all-ages total only)."""
    by_district = {}
    for workbook in sorted(D10_DIR.glob('*.XLSX')):
        workbook_state = ''
        for row in iter_sheet_rows(workbook, 'D-10'):
            area_name = str(row.get(3, '')).strip()
            if not workbook_state and looks_like_state_area_name(area_name):
                workbook_state = normalize_name(area_name)
            district_code = to_int(row.get(2))
            if district_code == 0:
                continue
            if not area_name or looks_like_state_area_name(area_name):
                continue
            if normalize_flag(row.get(4, '')) != 'TOTAL':
                continue
            if normalize_flag(row.get(5, '')) != 'ALL AGES':
                continue
            if normalize_flag(row.get(6, '')) != 'TOTAL':
                continue
            display_state = 'TELANGANA' if area_name.upper() in TELANGANA_DISTRICTS else workbook_state
            if display_state not in STATE_NAMES:
                continue
            key = (display_state, area_name, district_code)
            if key in by_district:
                continue
            by_district[key] = {
                'state': display_state, 'district': area_name, 'districtCode': district_code,
                'NeverMarried_Persons': to_int(row.get(10)), 'NeverMarried_Males': to_int(row.get(11)), 'NeverMarried_Females': to_int(row.get(12)),
                'CurrMarried_Persons':  to_int(row.get(13)), 'CurrMarried_Males':  to_int(row.get(14)), 'CurrMarried_Females':  to_int(row.get(15)),
                'Widowed_Persons':      to_int(row.get(16)), 'Widowed_Males':      to_int(row.get(17)), 'Widowed_Females':      to_int(row.get(18)),
                'Separated_Persons':    to_int(row.get(19)), 'Separated_Males':    to_int(row.get(20)), 'Separated_Females':    to_int(row.get(21)),
                'Divorced_Persons':     to_int(row.get(22)), 'Divorced_Males':     to_int(row.get(23)), 'Divorced_Females':     to_int(row.get(24)),
                'Unspecified_Persons':  to_int(row.get(25)), 'Unspecified_Males':  to_int(row.get(26)), 'Unspecified_Females':  to_int(row.get(27)),
            }
    output = sorted(by_district.values(), key=lambda r: (r['state'], r['district']))
    return output


print('Extract functions defined.')


## 6. Run ETL & write CSVs


In [ ]:
def write_csv(path: Path, rows: list, headers: list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='', encoding='utf-8') as fp:
        writer = csv.DictWriter(fp, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)


def safe_write(path: Path, rows: list, headers: list) -> str:
    try:
        write_csv(path, rows, headers)
        return f'  Saved  : {path.name}'
    except PermissionError as err:
        return f'  SKIPPED (locked): {path.name} — {err}'


# ── Run all five extractions ──────────────────────────────────────────────────
print('Extracting D02 (duration)...')
d02_rows = extract_d02_rows()

print('Extracting D03 (reason)...')
d03_rows = extract_d03_rows()

print('Extracting D04 (education)...')
d04_rows = extract_d04_rows()

print('Extracting D06 (economic activity)...')
d06_rows = extract_d06_rows()

print('Extracting D10 (marital status)...')
d10_rows = extract_d10_rows()

# ── Write to data-cleaning/ and react-app/public/ ────────────────────────────
print()
print('Writing CSVs...')
results = [
    safe_write(OUT_D02_CLEAN,  d02_rows, D02_HEADERS),
    safe_write(OUT_D02_PUBLIC, d02_rows, D02_HEADERS),
    safe_write(OUT_D03_CLEAN,  d03_rows, D03_HEADERS),
    safe_write(OUT_D03_PUBLIC, d03_rows, D03_HEADERS),
    safe_write(OUT_D04_CLEAN,  d04_rows, D04_HEADERS),
    safe_write(OUT_D04_PUBLIC, d04_rows, D04_HEADERS),
    safe_write(OUT_D06_CLEAN,  d06_rows, D06_HEADERS),
    safe_write(OUT_D06_PUBLIC, d06_rows, D06_HEADERS),
    safe_write(OUT_D10_CLEAN,  d10_rows, D10_HEADERS),
    safe_write(OUT_D10_PUBLIC, d10_rows, D10_HEADERS),
]
for r in results:
    print(r)


## 7. Validation summary


In [ ]:
print('=' * 55)
print('  ETL OUTPUT SUMMARY')
print('=' * 55)
for label, rows in [
    ('D02 Duration',         d02_rows),
    ('D03 Reason',           d03_rows),
    ('D04 Education',        d04_rows),
    ('D06 Economic Activity',d06_rows),
    ('D10 Marital Status',   d10_rows),
]:
    states    = len({r['state'] for r in rows})
    districts = len({(r['state'], r['district']) for r in rows})
    print(f'{label:<25} : {len(rows):>6,} rows  |  {states:>2} states  |  {districts:>3} districts')
